# Checking output files

In [3]:
from datetime import datetime, timedelta
import calendar
import collections
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
import sys

import glob
import pandas as pd
import xarray as xr
from IPython.display import display, HTML

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


my_dir = "/g/data/eg3/spr548/projects/"
sys.path.append(os.path.join(my_dir, "nesp_bff")+os.sep)
from nathers import location_details

# import utils
# from utils import locations, model_dict, vars_1hr, vars_day

In [7]:
step = "step3"
show_only_if_flagged = True

#==========================================
root_dir = "/g/data/eg3/nesp_bff/"
# location = "Sydney"
# scenario = "ssp370"
# model = "CESM2"
# time_period = "2041-2060"
vars_to_summarise_step2 = ["tas","huss","sfcWind","psl","uas","vas","clt","rsds","rsdsdir","rsdsdif"]
vars_to_summarise_step3 = ["tas","twbt","huss","psl","wind_speed","wind_dir","total_cloud_cover","rsds","rsdsdir","rsdsdif"]
locations = ["Darwin","Cairns","Brisbane","Longreach","Mildura","Adelaide","Perth","Sydney","Melbourne","Canberra","Hobart"]
vars_maybe_drop = ["time_offset", "round_method", "crs", "lat", "lon"]

input_dir = f"{root_dir}step3_calc_missing_vars/" if step == "step3" else f"{root_dir}step2_qdc_scaling/BARPA-R/"
vars_to_summarise = vars_to_summarise_step3 if step == "step3" else vars_to_summarise_step2

In [8]:
files = glob.glob(f"{input_dir}*.nc")
len(files)

462

In [9]:
qc_rows = []
errors = []

for file in sorted(files):
    base = file.split("/")[-1]
    parts = base.split("_")

    loc = parts[0] if len(parts) > 0 else None
    model_ = parts[2] if len(parts) > 2 else None
    ssp = parts[3] if len(parts) > 3 else None
    time_period_ = parts[9] if len(parts) > 9 else None

    header = f"{loc}: {model_}, {ssp}, {time_period_}"
    print(f"==================== {header} ====================")

    try:
        with xr.open_dataset(file) as da:
            df = (
                da.drop_vars([v for v in vars_maybe_drop if v in da.variables])
                  [vars_to_summarise]
                  .to_dataframe()
            )

        desc = df.describe()

        flagged = df.isna().any().any() or (df.nunique(dropna=True) <= 1).any()

        if (not show_only_if_flagged) or flagged:
            print("⚠️ Flagged" if flagged else "OK")
            display(HTML('<div style="overflow-x:auto; max-width:100%;">'))
            display(desc)
            display(HTML("</div>"))

        qc_rows.append({
            "file": base,
            "loc": loc,
            "model": model_,
            "ssp": ssp,
            "time_period": time_period_,
            "n_min": int(desc.loc["count"].min()),
            "any_nan": bool(df.isna().any().any()),
            "any_const": bool((df.nunique(dropna=True) <= 1).any()),
            "rsds_min": float(desc.loc["min", "rsds"]) if "rsds" in desc.columns else None,
            "rsds_max": float(desc.loc["max", "rsds"]) if "rsds" in desc.columns else None,
        })

    except Exception as e:
        errors.append({
            "file": base,
            "loc": loc,
            "model": model_,
            "ssp": ssp,
            "time_period": time_period_,
            "error": repr(e),
        })

qc_df = pd.DataFrame(qc_rows)
err_df = pd.DataFrame(errors)

print("\n=== QC summary (all files) ===")
display(qc_df)

if not err_df.empty:
    print("\n=== Errors ===")
    display(err_df)

==================== Adelaide: ACCESS-CM2, ssp126, 2021-2040 ====================
==================== Adelaide: ACCESS-CM2, ssp126, 2041-2060 ====================
⚠️ Flagged


,tas,twbt,huss,psl,wind_speed,wind_dir,total_cloud_cover,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175199.000000,175199.000000,175200.000000
mean,18.224072,12.929590,7.277942,101.227280,3.017486,7.487785,3.873362,202.196320,218.943024,64.824539
std,6.816612,3.682003,2.185331,0.728413,1.932686,4.646043,3.159390,293.196198,345.765442,100.089554
min,1.497444,0.693034,0.392788,97.528481,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,13.391983,10.324612,5.793806,100.749371,1.430837,4.000000,0.000000,0.000000,0.000000,0.000000
50%,16.970230,12.697818,7.048119,101.208359,2.935286,7.000000,4.000000,0.000000,0.000000,0.000000
75%,22.168592,15.318844,8.448796,101.695820,4.358454,11.000000,7.000000,345.881592,382.683289,92.500240
max,46.586071,28.978638,21.822971,103.618294,13.799576,16.000000,9.000000,1113.045410,1296.002319,723.055786


==================== Adelaide: ACCESS-CM2, ssp126, 2061-2080 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2021-2040 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2041-2060 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2061-2080 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2021-2040 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2041-2060 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2061-2080 ====================
⚠️ Flagged


,tas,twbt,huss,psl,wind_speed,wind_dir,total_cloud_cover,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175197.000000,175197.000000,175200.000000
mean,17.960510,12.628146,7.048107,101.196266,3.095285,7.617768,3.604355,201.684570,219.847733,64.121094
std,6.701255,3.438688,1.944658,0.715963,1.951415,4.674092,3.142279,292.196564,345.416138,97.803108
min,1.636778,0.822517,0.315108,97.727646,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,13.210593,10.201679,5.740314,100.711750,1.512857,4.000000,0.000000,0.000000,0.000000,0.000000
50%,16.638847,12.364718,6.849160,101.194939,3.056092,8.000000,3.000000,0.000000,0.000000,0.000000
75%,21.654992,14.855695,8.146292,101.690510,4.479830,11.000000,7.000000,341.344330,386.854431,92.647224
max,46.816719,27.362404,19.093647,103.256668,14.285907,16.000000,9.000000,1108.401978,1296.081299,721.370239


==================== Adelaide: ACCESS-ESM1-5, ssp370, 2021-2040 ====================
⚠️ Flagged


...

,tas,twbt,huss,psl,wind_speed,wind_dir,total_cloud_cover,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175199.000000,175199.000000,175200.000000
mean,19.207327,15.120675,9.296156,101.660057,5.372105,8.430582,3.620400,190.387894,204.485138,68.774078
std,5.181468,4.477241,3.360669,0.721320,2.918305,4.864906,3.248379,274.725586,329.255676,105.278641
min,3.026946,1.620766,1.098676,98.109329,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,15.576910,11.781052,6.606126,101.181717,3.156326,4.000000,0.000000,0.000000,0.000000,0.000000
50%,19.342810,15.324644,9.106777,101.699661,4.831289,8.000000,3.000000,0.000000,0.000000,0.000000
75%,22.689224,18.745608,11.794032,102.172440,7.275987,13.000000,7.000000,329.349182,332.648438,98.199814
max,46.727291,26.307487,19.798868,103.732849,83.812561,16.000000,9.000000,1111.323853,1292.421875,683.354492


==================== Sydney: ACCESS-ESM1-5, ssp370, 2041-2060 ====================
==================== Sydney: ACCESS-ESM1-5, ssp370, 2061-2080 ====================
==================== Sydney: CESM2, ssp126, 2021-2040 ====================
==================== Sydney: CESM2, ssp126, 2041-2060 ====================
⚠️ Flagged


,tas,twbt,huss,psl,wind_speed,wind_dir,total_cloud_cover,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175194.000000,175200.000000,175194.000000
mean,19.898195,15.804598,9.761570,101.721481,5.349599,8.469606,3.856809,192.762100,208.417465,68.738174
std,5.149637,4.403488,3.396461,0.718099,2.920070,4.641015,3.244185,278.453369,333.169128,105.569366
min,4.154095,2.660778,1.077956,98.551910,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,16.316440,12.540862,7.082072,101.243471,3.129902,5.000000,0.000000,0.000000,0.000000,0.000000
50%,20.097163,16.128542,9.660593,101.737701,4.801595,8.000000,4.000000,0.000000,0.000000,0.000000
75%,23.295516,19.390517,12.379400,102.217697,7.288040,12.000000,8.000000,333.757751,348.761421,96.453629
max,47.626766,27.095522,20.519081,104.112000,91.625565,16.000000,8.000000,1122.487793,1298.289673,711.197693


==================== Sydney: CESM2, ssp126, 2061-2080 ====================
==================== Sydney: CESM2, ssp370, 2021-2040 ====================
==================== Sydney: CESM2, ssp370, 2041-2060 ====================
==================== Sydney: CESM2, ssp370, 2061-2080 ====================
==================== Sydney: CMCC-ESM2, ssp126, 2021-2040 ====================
==================== Sydney: CMCC-ESM2, ssp126, 2041-2060 ====================
==================== Sydney: CMCC-ESM2, ssp126, 2061-2080 ====================
==================== Sydney: CMCC-ESM2, ssp370, 2021-2040 ====================
==================== Sydney: CMCC-ESM2, ssp370, 2041-2060 ====================
==================== Sydney: CMCC-ESM2, ssp370, 2061-2080 ====================
==================== Sydney: EC-Earth3, ssp126, 2021-2040 ====================
==================== Sydney: EC-Earth3, ssp126, 2041-2060 ====================
==================== Sydney: EC-Earth3, ssp126, 2061-2080 ==========

,file,loc,model,ssp,time_period,n_min,any_nan,any_const,rsds_min,rsds_max
0,Adelaide_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp126,2021-2040,175200,False,False,0.0,1125.402710
1,Adelaide_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp126,2041-2060,175199,True,False,0.0,1113.045410
2,Adelaide_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp126,2061-2080,175200,False,False,0.0,1120.985474
3,Adelaide_AUS-15_ACCESS-CM2_ssp370_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp370,2021-2040,175200,False,False,0.0,1120.141968
4,Adelaide_AUS-15_ACCESS-CM2_ssp370_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp370,2041-2060,175200,False,False,0.0,1106.313477
...,...,...,...,...,...,...,...,...,...,...
457,Sydney_AUS-15_NorESM2-MM_ssp126_r1i1p1f1_BOM_B...,Sydney,NorESM2-MM,ssp126,2041-2060,175200,False,False,0.0,1146.849243
458,Sydney_AUS-15_NorESM2-MM_ssp126_r1i1p1f1_BOM_B...,Sydney,NorESM2-MM,ssp126,2061-2080,175200,False,False,0.0,1128.617065
459,Sydney_AUS-15_NorESM2-MM_ssp370_r1i1p1f1_BOM_B...,Sydney,NorESM2-MM,ssp370,2021-2040,175200,False,False,0.0,1119.860840
460,Sydney_AUS-15_NorESM2-MM_ssp370_r1i1p1f1_BOM_B...,Sydney,NorESM2-MM,ssp370,2041-2060,175200,False,False,0.0,1127.152222


In [ ]:
# for file in files:
#     base = file.split('/')[-1]
#     loc = base.split('_')[0]
#     model = base.split('_')[2]
#     ssp = base.split('_')[3]
#     time_period = base.split('_')[9]
#     print(f"========================== {loc}: {model}, {ssp}, {time_period} ==========================")
#     da = xr.open_dataset(file)
#     print(da.drop_vars([v for v in ["time_offset","round_method",
#                                     "crs", "lat", "lon"] if v in da.variables])[vars_to_summarise].to_dataframe().describe())